# Week 7 — Wednesday Lab Notebook  
## Grouping & Combining (`groupby`, `aggregations`, `pivot tables`, `merge` / `join` / `concat`, `sorting`, `multi-index intuition`)

### Today’s Goal

By the end of today, students should be able to:

- group data by one or more columns
- calculate summaries like `sum`, `mean`, `count`, `min`, and `max`
- use `agg()` for multiple aggregations
- sort rows and summaries in a meaningful way
- build simple pivot tables
- combine datasets using `merge()`
- join tables using index-based joining
- stack data using `concat()`
- understand the basic idea of multi-index results
- prepare combined datasets for further analysis


## 3-Hour Structure

**0:00–0:20** Why grouping and combining matter  
**0:20–1:00** `groupby()` with single and multiple columns  
**1:00–1:25** Aggregations with `agg()` + sorting  
**1:25–1:50** Pivot tables  
**1:50–2:30** `merge()` and join types  
**2:30–2:45** `concat()` and index-based `join()`  
**2:45–2:55** Multi-index intuition  
**2:55–3:00** Recap + task briefing


## Part A — Why Grouping and Combining Matter

In real life, we usually do not want to see only raw rows.

We often want questions like:

- what is the total sale in each city?
- which department has the highest average salary?
- how many students are there in each section?
- what is the total quantity of each product?
- how do we combine customer data with order data?

These questions are answered using:

- `groupby()` for summaries
- `pivot_table()` for structured reports
- `merge()` / `join()` / `concat()` for combining data

These are core data analysis skills.


## Part B — Import Libraries


In [ ]:
import pandas as pd
import numpy as np


## Part C — Create a Small Sales Dataset

We will start with one dataset and use it for many examples.


In [ ]:
sales = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "city": ["Lahore", "Karachi", "Lahore", "Islamabad", "Karachi", "Lahore", "Islamabad", "Karachi", "Lahore", "Karachi"],
    "category": ["Electronics", "Clothing", "Clothing", "Electronics", "Clothing", "Electronics", "Clothing", "Electronics", "Clothing", "Clothing"],
    "salesperson": ["Ali", "Sara", "Ali", "Ahmed", "Sara", "Ali", "Ahmed", "Sara", "Ali", "Sara"],
    "quantity": [2, 4, 3, 1, 5, 2, 2, 1, 4, 3],
    "price": [50000, 3000, 2500, 70000, 3200, 52000, 2800, 68000, 2600, 3500]
})

sales["revenue"] = sales["quantity"] * sales["price"]
sales


### What can we analyze from this dataset?

We can answer questions like:

- total revenue by city
- average quantity by category
- number of orders by salesperson
- revenue by city and category
- which city has more orders

This is exactly where grouping starts.


## Part D — First Inspect the Dataset


In [ ]:
print(sales.head())


In [ ]:
print("Shape:", sales.shape)
print("Columns:", list(sales.columns))


In [ ]:
sales.info()


## Part E — Group by One Column

The most basic use of `groupby()` is to split data into groups based on one column.


In [ ]:
sales.groupby("city")


The command above creates a grouped object.

It does not show final results yet.

To get useful output, we apply an aggregation like `sum()` or `mean()`.


## Part F — Total Revenue by City


In [ ]:
sales.groupby("city")["revenue"].sum()


### What happened here?

- `groupby("city")` made groups for each city
- `["revenue"]` selected the revenue column
- `.sum()` added revenue inside each group

So now we know total revenue for each city.


## Part G — Average Quantity by Category


In [ ]:
sales.groupby("category")["quantity"].mean()


This gives the average quantity for each product category.

This is useful when we want group-wise averages instead of totals.


## Part H — Count Orders by Salesperson


In [ ]:
sales.groupby("salesperson")["order_id"].count()


### `count()` vs `size()`

Both are used for counting, but there is a small difference:

- `count()` counts non-missing values in a selected column
- `size()` counts total rows in each group

In many clean datasets, both may give the same result.


In [ ]:
sales.groupby("salesperson").size()


## Part I — Multiple Aggregations on One Column

Sometimes one summary is not enough.

We may want total, average, minimum, and maximum together.


In [ ]:
sales.groupby("city")["revenue"].agg(["sum", "mean", "min", "max"])


This is a compact summary table.

It is very useful in reports.


## Part J — Aggregate Multiple Columns Together


In [ ]:
sales.groupby("city").agg({
    "quantity": ["sum", "mean"],
    "revenue": ["sum", "mean"],
    "order_id": "count"
})


### Why does the output look different?

Because we asked for multiple columns and multiple calculations.

So Pandas created a wider summary table.


## Part K — Group by More Than One Column

We can also group by two or more columns.


In [ ]:
sales.groupby(["city", "category"])["revenue"].sum()


Now each result is based on a pair:

- city
- category

So this gives revenue for each category inside each city.


## Part L — Convert Grouped Result into a Simple Table


In [ ]:
sales.groupby(["city", "category"])["revenue"].sum().reset_index()


### Why use `reset_index()`?

Without `reset_index()`, the grouped columns become index levels.

With `reset_index()`, they come back as normal columns.

This is usually easier for beginners to read.


## Part M — Use `as_index=False` from the Start


In [ ]:
sales.groupby(["city", "category"], as_index=False)["revenue"].sum()


This is another clean way to get tabular output directly.


## Part N — Sort Summary Results

After grouping, we often sort the output to find highest or lowest values.


In [ ]:
city_revenue = sales.groupby("city", as_index=False)["revenue"].sum()
city_revenue


In [ ]:
city_revenue.sort_values(by="revenue", ascending=False)


### Why is sorting important?

Sorting helps answer questions like:

- which city performed best?
- which category sold least?
- who has the highest total orders?

Grouped results become more useful after sorting.


## Part O — Sort by More Than One Column


In [ ]:
sales.sort_values(by=["city", "revenue"], ascending=[True, False])


This sorts first by city name and then, inside each city, by revenue.


## Part P — Named Aggregations for Cleaner Column Names

Sometimes the default column names are not very clean.

We can give custom names.


In [ ]:
sales.groupby("city").agg(
    total_revenue=("revenue", "sum"),
    avg_quantity=("quantity", "mean"),
    order_count=("order_id", "count")
).reset_index()


This format is easier to read and very good for class demos.


## Part Q — Pivot Table Basics

A pivot table is another way to summarize grouped data.

It often feels like an Excel-style summary.


In [ ]:
pd.pivot_table(
    sales,
    values="revenue",
    index="city",
    aggfunc="sum"
)


This gives almost the same idea as `groupby("city")["revenue"].sum()`.

But pivot tables become very powerful when we use both rows and columns.


## Part R — Pivot Table with Rows and Columns


In [ ]:
pd.pivot_table(
    sales,
    values="revenue",
    index="city",
    columns="category",
    aggfunc="sum"
)


### What does this show?

- rows are cities
- columns are categories
- values are total revenue

This creates a compact report.


## Part S — Fill Missing Pivot Values


In [ ]:
pd.pivot_table(
    sales,
    values="revenue",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0
)


`fill_value=0` is useful when some group combinations do not exist.


## Part T — Pivot Table with Multiple Aggregations


In [ ]:
pd.pivot_table(
    sales,
    values="revenue",
    index="city",
    columns="category",
    aggfunc=["sum", "mean"],
    fill_value=0
)


This creates a wider report.

It is useful, but beginners should read it slowly because it becomes more detailed.


## Part U — Create Customer and Order Datasets for Merge

Now let us learn how to combine tables.

We will make two related datasets.


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "customer_name": ["Ayesha", "Bilal", "Hina", "Usman", "Zara"],
    "city": ["Lahore", "Karachi", "Lahore", "Islamabad", "Karachi"]
})

orders = pd.DataFrame({
    "order_id": [201, 202, 203, 204, 205, 206],
    "customer_id": [1, 2, 2, 3, 6, 1],
    "product": ["Laptop", "Shoes", "Bag", "Phone", "Watch", "Keyboard"],
    "amount": [120000, 5000, 3500, 90000, 8000, 6000]
})

customers


In [ ]:
orders


## Part V — Inner Merge

An inner merge keeps only matching keys from both tables.


In [ ]:
pd.merge(customers, orders, on="customer_id", how="inner")


### What happened?

Only those rows stayed where `customer_id` exists in both tables.

So unmatched records are removed.


## Part W — Left Merge

A left merge keeps all rows from the left table.


In [ ]:
pd.merge(customers, orders, on="customer_id", how="left")


This keeps all customers.

If a customer has no matching order, order columns become `NaN`.


## Part X — Right Merge


In [ ]:
pd.merge(customers, orders, on="customer_id", how="right")


This keeps all orders.

Notice that `customer_id = 6` exists in orders but not in customers.

So customer information becomes missing for that row.


## Part Y — Outer Merge

An outer merge keeps all rows from both sides.


In [ ]:
pd.merge(customers, orders, on="customer_id", how="outer")


Outer merge is useful when we do not want to lose any record.


## Part Z — Merge Indicator

Sometimes we want to know where each row came from.


In [ ]:
pd.merge(customers, orders, on="customer_id", how="outer", indicator=True)


The `_merge` column tells us whether a row came from:

- left only
- right only
- both


## Part AA — Merge on Different Column Names

Sometimes the key columns have different names in two tables.


In [ ]:
students = pd.DataFrame({
    "std_id": [1, 2, 3],
    "name": ["Ali", "Sara", "Ahmed"]
})

marks = pd.DataFrame({
    "student_id": [1, 2, 4],
    "marks": [85, 90, 76]
})

pd.merge(students, marks, left_on="std_id", right_on="student_id", how="outer")


This is useful when datasets are related but column names do not match exactly.


## Part AB — Concatenate Rows

`concat()` is used when we want to attach tables one below another or side by side.


In [ ]:
semester1 = pd.DataFrame({
    "student_id": [1, 2, 3],
    "name": ["Ali", "Sara", "Ahmed"]
})

semester2 = pd.DataFrame({
    "student_id": [4, 5],
    "name": ["Hina", "Usman"]
})

pd.concat([semester1, semester2], axis=0)


### Why do we sometimes see repeated indexes?

Because original indexes are kept by default.


In [ ]:
pd.concat([semester1, semester2], axis=0, ignore_index=True)


`ignore_index=True` makes the row numbers clean again.


## Part AC — Concatenate Columns


In [ ]:
personal = pd.DataFrame({
    "name": ["Ali", "Sara", "Ahmed"]
})

academic = pd.DataFrame({
    "marks": [85, 90, 78],
    "grade": ["A", "A", "B"]
})

pd.concat([personal, academic], axis=1)


Here the tables are attached side by side.


## Part AD — Join Using Index

`join()` is similar to merge, but it works naturally with indexes.


In [ ]:
left_table = pd.DataFrame({
    "name": ["Ali", "Sara", "Ahmed"]
}, index=[1, 2, 3])

right_table = pd.DataFrame({
    "marks": [85, 90, 78]
}, index=[1, 2, 4])

left_table


In [ ]:
right_table


In [ ]:
left_table.join(right_table, how="left")


### Why learn `join()`?

Because many real datasets use indexes, and `join()` becomes convenient in such cases.


## Part AE — Multi-Index Intuition

A multi-index means more than one index level.

This often happens after grouping by multiple columns.


In [ ]:
grouped_multi = sales.groupby(["city", "category"])["revenue"].sum()
grouped_multi


Notice the result index is not just one label.

It has two levels:

- city
- category


In [ ]:
grouped_multi.index


## Part AF — Convert Multi-Index Result Back to Normal Columns


In [ ]:
grouped_multi.reset_index()


This is often the easiest way for beginners to handle multi-index output.


## Part AG — Small Real-Life Style Example 1: Student Results Summary

Let us make a student marks dataset and summarize it.


In [ ]:
results = pd.DataFrame({
    "section": ["A", "A", "A", "B", "B", "B", "C", "C"],
    "subject": ["Math", "English", "Math", "Math", "English", "Math", "English", "Math"],
    "student": ["Ali", "Sara", "Ahmed", "Hina", "Usman", "Zara", "Hamza", "Noor"],
    "marks": [85, 78, 92, 74, 88, 69, 91, 81]
})

results


In [ ]:
results.groupby("section")["marks"].mean().reset_index()


In [ ]:
results.groupby(["section", "subject"])["marks"].agg(["mean", "max", "min"]).reset_index()


In [ ]:
pd.pivot_table(
    results,
    values="marks",
    index="section",
    columns="subject",
    aggfunc="mean",
    fill_value=0
)


### What did we learn from this example?

- we grouped by section
- then by section and subject
- then converted the same idea into a pivot table

This is a very common education-data workflow.


## Part AH — Small Real-Life Style Example 2: Combine Product and Stock Data


In [ ]:
products = pd.DataFrame({
    "product_id": [11, 12, 13, 14],
    "product_name": ["Mouse", "Keyboard", "Monitor", "Speaker"],
    "category": ["Accessories", "Accessories", "Display", "Audio"]
})

stock = pd.DataFrame({
    "product_id": [11, 12, 14, 15],
    "stock_qty": [50, 40, 20, 10]
})

products


In [ ]:
stock


In [ ]:
pd.merge(products, stock, on="product_id", how="left")


In [ ]:
pd.merge(products, stock, on="product_id", how="outer", indicator=True)


This shows which products have stock data and which do not.


## Part AI — Common Beginner Mistakes

Students often make these mistakes:

1. forgetting to choose a column before aggregation  
2. using `groupby()` but not applying `sum()` / `mean()` / `count()`  
3. getting confused when grouped columns become indexes  
4. forgetting `reset_index()`  
5. mixing up `merge()` and `concat()`  
6. expecting `concat()` to match rows like a database join  
7. using the wrong join type in `merge()`  
8. not sorting grouped summaries before interpretation  
9. reading pivot tables too quickly without checking rows and columns  
10. getting scared of multi-index output even though `reset_index()` can simplify it


## Part AJ — Mini Practice in Class

Try these small exercises before moving to the after-lab tasks:

1. find total quantity by city  
2. find average revenue by category  
3. group sales by city and category  
4. sort city revenue from high to low  
5. create a pivot table of quantity by city and category  
6. merge customers and orders using left join  
7. concatenate two small student tables  
8. convert a grouped multi-index result back into normal columns


## Part AK — Recap

Today we learned how to:

- group data by one or more columns
- calculate sums, averages, counts, minimums, and maximums
- use `agg()` for multiple summaries
- sort grouped output
- create pivot tables
- combine tables using `merge()`
- join index-based tables
- stack tables using `concat()`
- understand and simplify multi-index output

These tools are used again and again in real data analysis.


# After-Lab Tasks (10)

### Task 1
Create a DataFrame of 8 sales records with columns: `city`, `product`, `quantity`, and `price`. Create a new `revenue` column.

### Task 2
Group the dataset by `city` and find the total revenue for each city.

### Task 3
Group the dataset by `product` and find the average quantity.

### Task 4
Group by both `city` and `product` and find total revenue.

### Task 5
Use `agg()` to calculate `sum`, `mean`, and `max` for the `revenue` column by city.

### Task 6
Sort the grouped revenue result from highest to lowest.

### Task 7
Create a pivot table that shows total `quantity` for each `city` and `product`.

### Task 8
Create two related DataFrames such as `students` and `marks`, then merge them using an inner join.

### Task 9
Create two small DataFrames with the same columns and combine them using `pd.concat()`.

### Task 10
Create a grouped result using two columns and then use `reset_index()` to convert the multi-index result into a normal table.


# Optional Homework Challenge

Create a small business dataset of your own with at least:

- 10 rows
- 2 cities
- 3 product categories
- quantity and price columns
- one customer table and one order table

Then do all of the following:

1. create a `revenue` column  
2. group by city and find total revenue  
3. group by city and category together  
4. build one pivot table  
5. merge customer and order data  
6. sort your final summary  
7. write 5 to 7 lines explaining what insights you found
